# Vue/San 批量组件生产后的问题、修复与研究启示

**记录日期：** 2026-09-02  
**数据范围：** 本轮新增并逐一人工检查的 15 对 Vue/San 组件（simple、medium、complex 各 5 对）  
**记录目的：** 汇总数据生产完成后才暴露的问题、根因、修复方式和验证结果，为后续论文中的数据质量控制、迁移难点分析和有效性威胁讨论提供素材。

> 本记录中的“通过”仅表示用户在两个手工测试页中逐个检查并确认；它不等同于自动化测试或大规模浏览器兼容性测试。

## 1. 本批组件清单

| 复杂度 | 组件 |
|---|---|
| Simple | `StatusSwitch`、`TextLengthGauge`、`DeliveryPhaseCard`、`ChoiceChips`、`DisclosureNotice` |
| Medium | `ReservationForm`、`ComparisonTable`、`MilestoneTimeline`、`QuizNavigator`、`CartSummary` |
| Complex | `RuleBuilder`、`AccessMatrix`、`ForecastSimulator`、`MediaReviewStudio`、`TopologyEditor` |

最终状态：15 对组件均已完成逐项人工检查。测试页最后保留的组件是 `TopologyEditor`。

In [ ]:
from collections import Counter

components = {
    'simple': [
        'StatusSwitch', 'TextLengthGauge', 'DeliveryPhaseCard',
        'ChoiceChips', 'DisclosureNotice'
    ],
    'medium': [
        'ReservationForm', 'ComparisonTable', 'MilestoneTimeline',
        'QuizNavigator', 'CartSummary'
    ],
    'complex': [
        'RuleBuilder', 'AccessMatrix', 'ForecastSimulator',
        'MediaReviewStudio', 'TopologyEditor'
    ],
}

print('组件对总数:', sum(map(len, components.values())))
print('复杂度分布:', {level: len(names) for level, names in components.items()})

## 2. 生产后暴露的问题总览

本轮共记录 6 类已处理问题，其中 1 类属于运行环境，5 类属于 San 模板运行时兼容性。5 个出现模板问题的组件占本批组件的 **33.3%（5/15）**；问题集中在 medium 和 complex 组件，说明仅靠文件存在性、语法形态和元数据一致性检查，不能替代真实运行时验证。

| 编号 | 层面/组件 | 人工测试现象 | 根因 | 解决办法 | 状态 |
|---|---|---|---|---|---|
| P0 | 运行环境 | 无法在 Node 环境中直接加载 San 运行时 | 项目未声明 `san` npm 依赖 | 安装并在 `package.json` 中声明 `san@^3.15.6`，同步更新锁文件 | 已解决 |
| P1 | `ComparisonTable` | San 初始状态下 3 个方案复选框均未勾选 | 模板内直接调用 `selectedIds.indexOf(...)`，超出 San 模板表达式支持范围 | 将判断封装为组件方法 `isSelected(id, selectedIds)`，模板只调用该方法 | 已解决并人工确认 |
| P2 | `CartSummary` | 初始金额区域为空，看起来像初始数据未加载 | 模板内直接调用数值的 `.toFixed(2)` 导致插值求值失败 | 新增 `formatMoney(value)`，把格式化放到组件 JavaScript 方法中 | 已解决并人工确认 |
| P3 | `ForecastSimulator` | 已保存情景无数据、对比按钮无效，累计收入等汇总值为空 | 模板同时使用 `compareIds.indexOf(...)` 和 `value.toLocaleString(...)` | 分别封装为 `isCompared(...)` 与 `formatNumber(...)`，模板只使用方法返回值 | 已解决并人工确认 |
| P4 | `MediaReviewStudio` | 右上角审阅人头像文字未显示 | 模板直接调用 `reviewer.name.slice(0, 1)` | 新增 `reviewerInitial(name)`，在组件方法内部完成字符串截取 | 已解决并人工确认 |
| P5 | `TopologyEditor` | 右上角缩放百分比数字未显示 | 模板直接调用全局对象 `Math.round(...)` | 新增 `zoomPercent(zoom)`，在组件方法内部完成计算 | 已解决并人工确认 |

In [ ]:
issues = [
    {'id': 'P0', 'component': None, 'level': 'environment', 'cause': 'missing_dependency'},
    {'id': 'P1', 'component': 'ComparisonTable', 'level': 'medium', 'cause': 'array_member_call'},
    {'id': 'P2', 'component': 'CartSummary', 'level': 'medium', 'cause': 'number_member_call'},
    {'id': 'P3', 'component': 'ForecastSimulator', 'level': 'complex', 'cause': 'array_and_number_member_calls'},
    {'id': 'P4', 'component': 'MediaReviewStudio', 'level': 'complex', 'cause': 'string_member_call'},
    {'id': 'P5', 'component': 'TopologyEditor', 'level': 'complex', 'cause': 'global_object_call'},
]

component_issues = [item for item in issues if item['component']]
print('组件级问题数:', len(component_issues))
print('受影响组件比例: {:.1%}'.format(len(component_issues) / 15))
print('按复杂度分布:', dict(Counter(item['level'] for item in component_issues)))

## 3. 根因分析

### 3.1 数据为空只是表象，真正失败的是模板表达式求值

`ComparisonTable` 的 `selectedIds`、`CartSummary` 的商品与金额、`ForecastSimulator` 的预测序列等数据事实上已经在 `initData`、`inited` 或 computed 中生成。页面空白并不是初始化生命周期没有执行，而是某个插值或属性绑定表达式无法被 San 模板解析/执行，最终造成局部内容空白、选中态丢失或点击反馈无效。

这一区分很重要：若按“初始化失败”修复，很可能重复修改数据模型，却无法消除视图层错误。正确的诊断顺序应为：**确认数据状态 → 定位首个模板求值错误 → 将复杂运算移出模板 → 再验证交互链路**。

### 3.2 Vue 模板可用写法不能机械复制到 San

本批问题具有同一模式：Vue 模板中常见的成员方法或全局对象调用，被直接迁移到了 San 模板。San 模板表达式不是完整 JavaScript 环境；数组的 `indexOf`、字符串的 `slice`、数字的 `toFixed`/`toLocaleString` 以及全局 `Math` 调用均不应直接出现在模板中。

统一修复原则是：**模板只负责声明性绑定，格式化、集合判断和数值运算放入组件方法或 computed。** 组件脚本内部仍可正常使用标准 JavaScript API。

### 3.3 一个不兼容表达式可能破坏一整段业务语义

`ForecastSimulator` 同时表现为“已保存情景无数据”“点击无效”“累计收入为空”。这些现象并非三个互不相关的缺陷，而是格式化与选中判断两个模板表达式错误沿多个渲染位置传播的结果。复杂组件的错误影响范围通常大于报错位置，因此验证时必须覆盖初始展示、状态变更和派生统计三个阶段。

## 4. 可复用的修复模式

| 不建议出现在 San 模板中的表达式 | 模板中的替代写法 | 组件脚本方法 |
|---|---|---|
| `selectedIds.indexOf(id) >= 0` | `isSelected(id, selectedIds)` | `return selectedIds.indexOf(id) >= 0` |
| `value.toFixed(2)` | `formatMoney(value)` | `return Number(value).toFixed(2)` |
| `value.toLocaleString('zh-CN')` | `formatNumber(value)` | `return Number(value).toLocaleString('zh-CN')` |
| `name.slice(0, 1)` | `reviewerInitial(name)` | `return String(name || '').slice(0, 1)` |
| `Math.round(zoom * 100)` | `zoomPercent(zoom)` | `return Math.round(Number(zoom) * 100)` |

Vue 与 San 组件对需要同步保留同名辅助方法，使两份实现的模板语义和公开行为仍然对应，而不是只对 San 做无法追踪的特例补丁。

## 5. 验证过程与结果

本轮采用串行人工验证：每次只把一对组件写入 `tests/manual/vue-test-runner.html` 和 `tests/manual/san-test-runner.html`，对照检查初始视觉状态和主要交互；用户确认后才切换到下一个组件。发现问题时先停留在当前组件，修复 Vue/San 源文件并重新检查。

最终检查结果：

- 15 对组件均已由用户逐项确认。
- 5 个受影响 San 模板中不再直接出现 `.indexOf(`、`.slice(`、`.toFixed(`、`.toLocaleString(` 或 `Math.`。
- `san@3.15.6` 已安装，可用于 San 组件运行时求值检查。
- `git diff --check` 已通过；仅出现换行符风格提示。
- 两个手工测试页当前均指向 `TopologyEditor`。

In [ ]:
from pathlib import Path
import re

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'data' / 'datasets' / 'components').exists():
            return candidate
    raise FileNotFoundError('未找到项目根目录')

project_root = find_project_root()
san_files = []
for level, names in components.items():
    level_dir = {'simple': '01_simple', 'medium': '02_medium', 'complex': '03_complex'}[level]
    for name in names:
        san_files.append(
            project_root / 'data' / 'datasets' / 'components' / level_dir / name / 'san' / f'{name}.san'
        )

risk_patterns = {
    'array member call': r'\.indexOf\s*\(',
    'string slice': r'\.slice\s*\(',
    'fixed decimals': r'\.toFixed\s*\(',
    'locale formatting': r'\.toLocaleString\s*\(',
    'global Math access': r'\bMath\.',
}

findings = []
for san_file in san_files:
    source = san_file.read_text(encoding='utf-8')
    template_match = re.search(r'<template>(.*?)</template>', source, flags=re.S)
    template = template_match.group(1) if template_match else ''
    for label, pattern in risk_patterns.items():
        if re.search(pattern, template):
            findings.append((san_file.stem, label))

print('已扫描 San 模板数:', len(san_files))
print('高风险模板表达式命中:', findings or '无')

## 6. 修复后仍待工程化解决的问题

### R1. 静态验证存在假阴性

这 5 个模板问题在生产阶段的静态检查中均未被拦截，却在浏览器人工检查中出现。上面的正则扫描可以覆盖本次已知模式，但它仍是启发式规则，不能替代 San 模板解析器或真实渲染测试。后续应把“模板编译 + 挂载 + 控制台错误捕获”加入批量生产门禁。

### R2. 人工检查缺少可重复的自动化回归

逐组件人工确认能够发现视觉与交互问题，但成本高，且修复后容易在后续批次中复发。建议为每对组件保存最小交互脚本，至少覆盖：初始数据可见、默认选中态、一次主要点击、一次输入更新和关键派生值。Vue/San 应执行相同动作并比较可观察结果。

### R3. 验证证据与元数据状态尚未完全同步

当前 `migration_notes.json` 中本批条目的 `visual_test` 仍多为 `not_run`，与本轮已经完成的人工确认不一致。论文统计前应设计统一的验证记录结构，至少包含测试方式、测试日期、执行者、结果和修复版本，避免把人工通过、静态通过和自动化通过混为一类。

## 7. 可用于论文的归纳

### 数据质量控制结论

1. 成对组件的数据质量不能仅由文件结构、组件名称、复杂度标签和静态语法判断，需要加入目标框架的真实运行时验证。
2. 跨框架迁移的主要风险不只在指令名称映射，还在模板表达式子语言的能力边界。
3. 同一底层兼容性错误可表现为数据空白、默认状态错误和交互无效等多种表层现象，诊断时应按数据层、表达式层、DOM 层和交互层分层排查。
4. 将复杂表达式收敛到组件方法，可以降低模板语言差异，并形成可被规则化、自动检测的迁移模式。

### 可报告的量化结果

- 样本规模：15 对 Vue/San 组件。
- 人工检查覆盖率：100%（15/15）。
- 生产后发现模板运行时问题的组件：5 个，占 33.3%。
- 按复杂度分布：medium 2 个，complex 3 个，simple 0 个。
- 已知问题修复后的人工复核通过率：100%（5/5）。

### 有效性威胁

本批样本量较小，且由人工检查完成，可能受测试路径覆盖不足和主观判断影响；33.3% 只能描述当前批次，不能直接外推为所有 Vue→San 迁移任务的缺陷率。后续应扩大样本量、固定测试协议，并用自动化运行结果与人工结果交叉验证。

## 8. 相关项目文件

- 数据集清单：`data/datasets/dataset_manifest.json`
- 迁移说明：`data/datasets/features/migration_notes.json`
- 数据生产规则：`skills/vue-san-dataset-production/SKILL.md`
- 人工测试配置规则：`skills/manual-runner-config/SKILL.md`
- Vue 测试页：`tests/manual/vue-test-runner.html`
- San 测试页：`tests/manual/san-test-runner.html`